# Random Forest con Alpha Vantage

En este notebook voy a entrenar un Random Forest para comprobar si existen relaciones no lineales entre las variables predictoras

Primero evaluaré una configuración base. Después ajustaré sus parámetros utilizando únicamente divisiones temporales dentro del conjunto de entrenamiento. El conjunto de prueba final no se utilizará en esta etapa

In [1]:
from pathlib import Path

import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    TimeSeriesSplit
)

# Localizo la carpeta principal
ruta_actual = Path.cwd().resolve()

if ruta_actual.name == "notebooks":
    ruta_proyecto = ruta_actual.parent
else:
    ruta_proyecto = ruta_actual

archivo_particiones = (
    ruta_proyecto
    / "data"
    / "processed"
    / "eurusd_alpha_vantage_particiones.csv"
)

datos = pd.read_csv(
    archivo_particiones,
    parse_dates=["Date"],
    index_col="Date"
)

datos = datos.sort_index()
datos["Objetivo"] = datos["Objetivo"].astype("Int64")

print("Dimensiones:", datos.shape)

Dimensiones: (4980, 21)


In [2]:
variables_predictoras = [
    "Retorno_diario",
    "Retorno_lag_1",
    "Retorno_lag_2",
    "Retorno_lag_3",
    "Retorno_lag_5",
    "Rango_diario",
    "Cuerpo_vela",
    "Posicion_cierre",
    "Distancia_MA5",
    "Distancia_MA10",
    "Distancia_MA20",
    "Volatilidad_5",
    "Volatilidad_20",
    "RSI_14",
    "MACD_hist"
]

# Utilizo únicamente entrenamiento y validación
entrenamiento = datos[
    datos["Particion"] == "entrenamiento"
].copy()

validacion = datos[
    datos["Particion"] == "validacion"
].copy()

X_entrenamiento = entrenamiento[variables_predictoras]
y_entrenamiento = entrenamiento["Objetivo"].astype(int)

X_validacion = validacion[variables_predictoras]
y_validacion = validacion["Objetivo"].astype(int)

print("Entrenamiento:", X_entrenamiento.shape)
print("Validación:", X_validacion.shape)

Entrenamiento: (4052, 15)
Validación: (522, 15)


In [3]:
# Utilizo la misma funcion

def calcular_metricas(
    nombre,
    particion,
    y_real,
    y_predicho,
    probabilidades
):
    return {
        "Modelo": nombre,
        "Particion": particion,
        "Accuracy": accuracy_score(
            y_real,
            y_predicho
        ),
        "Balanced_accuracy": balanced_accuracy_score(
            y_real,
            y_predicho
        ),
        "Precision": precision_score(
            y_real,
            y_predicho,
            zero_division=0
        ),
        "Recall": recall_score(
            y_real,
            y_predicho,
            zero_division=0
        ),
        "F1": f1_score(
            y_real,
            y_predicho,
            zero_division=0
        ),
        "ROC_AUC": roc_auc_score(
            y_real,
            probabilidades
        )
    }


resultados_rf = []

## Configuración base

La primera versión utiliza una configuración cercana a la estándar. Servirá como referencia para comprobar cuánto cambia el modelo después del ajuste

Los árboles pueden adaptarse demasiado al entrenamiento, por lo que compararé siempre entrenamiento y validación

In [4]:
modelo_rf_base = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

modelo_rf_base.fit(
    X_entrenamiento,
    y_entrenamiento
)

pred_rf_base_entrenamiento = modelo_rf_base.predict(
    X_entrenamiento
)

pred_rf_base_validacion = modelo_rf_base.predict(
    X_validacion
)

prob_rf_base_entrenamiento = modelo_rf_base.predict_proba(
    X_entrenamiento
)[:, 1]

prob_rf_base_validacion = modelo_rf_base.predict_proba(
    X_validacion
)[:, 1]

resultados_rf.append(
    calcular_metricas(
        "Random Forest base",
        "Entrenamiento",
        y_entrenamiento,
        pred_rf_base_entrenamiento,
        prob_rf_base_entrenamiento
    )
)

resultados_rf.append(
    calcular_metricas(
        "Random Forest base",
        "Validación",
        y_validacion,
        pred_rf_base_validacion,
        prob_rf_base_validacion
    )
)

## Ajuste de parámetros con validación temporal

Voy a buscar una configuración más controlada utilizando únicamente el conjunto de entrenamiento

Las divisiones respetarán el orden cronológico. También dejaré una fila de separación entre cada bloque de entrenamiento y su bloque de validación interna, porque el objetivo de cada fila depende de la jornada siguiente

In [5]:
# Creo divisiones internas respetando el orden temporal
validacion_temporal = TimeSeriesSplit(
    n_splits=5,
    gap=1
)

modelo_rf_busqueda = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

parametros_rf = {
    "n_estimators": [300, 500, 800],
    "max_depth": [3, 5, 8, 12, None],
    "min_samples_leaf": [2, 5, 10, 20],
    "max_features": ["sqrt", 0.5, 1.0],
    "class_weight": [
        None,
        "balanced_subsample"
    ]
}

busqueda_rf = RandomizedSearchCV(
    estimator=modelo_rf_busqueda,
    param_distributions=parametros_rf,
    n_iter=30,
    scoring="balanced_accuracy",
    cv=validacion_temporal,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    refit=True
)

busqueda_rf.fit(
    X_entrenamiento,
    y_entrenamiento
)

modelo_rf_ajustado = busqueda_rf.best_estimator_

print("Mejor balanced accuracy interna:")
print(round(busqueda_rf.best_score_, 4))

print("\nMejores parámetros:")
print(busqueda_rf.best_params_)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Mejor balanced accuracy interna:
0.5135

Mejores parámetros:
{'n_estimators': 800, 'min_samples_leaf': 10, 'max_features': 0.5, 'max_depth': 8, 'class_weight': 'balanced_subsample'}


In [6]:
pred_rf_ajustado_entrenamiento = (
    modelo_rf_ajustado.predict(
        X_entrenamiento
    )
)

pred_rf_ajustado_validacion = (
    modelo_rf_ajustado.predict(
        X_validacion
    )
)

prob_rf_ajustado_entrenamiento = (
    modelo_rf_ajustado.predict_proba(
        X_entrenamiento
    )[:, 1]
)

prob_rf_ajustado_validacion = (
    modelo_rf_ajustado.predict_proba(
        X_validacion
    )[:, 1]
)

resultados_rf.append(
    calcular_metricas(
        "Random Forest ajustado",
        "Entrenamiento",
        y_entrenamiento,
        pred_rf_ajustado_entrenamiento,
        prob_rf_ajustado_entrenamiento
    )
)

resultados_rf.append(
    calcular_metricas(
        "Random Forest ajustado",
        "Validación",
        y_validacion,
        pred_rf_ajustado_validacion,
        prob_rf_ajustado_validacion
    )
)

tabla_resultados_rf = pd.DataFrame(
    resultados_rf
)

columnas_metricas = [
    "Accuracy",
    "Balanced_accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC_AUC"
]

tabla_resultados_rf[columnas_metricas] = (
    tabla_resultados_rf[columnas_metricas]
    .round(4)
)

display(
    tabla_resultados_rf.set_index(
        ["Modelo", "Particion"]
    )
)

Accuracy  Balanced_accuracy  Precision  \
Modelo                 Particion                                               
Random Forest base     Entrenamiento    1.0000             1.0000     1.0000   
                       Validación       0.4713             0.4717     0.4635   
Random Forest ajustado Entrenamiento    0.8231             0.8228     0.8089   
                       Validación       0.4751             0.4762     0.4690   

                                      Recall      F1  ROC_AUC  
Modelo                 Particion                               
Random Forest base     Entrenamiento  1.0000  1.0000   1.0000  
                       Validación     0.4961  0.4792   0.4805  
Random Forest ajustado Entrenamiento  0.8497  0.8288   0.9074  
                       Validación     0.5312  0.4982   0.4756

## Estabilidad anual en validación

Además del resultado conjunto, voy a evaluar 2023 y 2024 por separado. Esto permitirá comprobar si el rendimiento del Random Forest se mantiene o si vuelve a concentrarse en un único año

In [7]:
# Identifico el año de la jornada que se intenta predecir
fecha_objetivo = (
    datos.index
    .to_series()
    .shift(-1)
)

anio_objetivo_validacion = (
    fecha_objetivo
    .loc[validacion.index]
    .dt.year
)

pred_rf_validacion_serie = pd.Series(
    pred_rf_ajustado_validacion,
    index=validacion.index
)

prob_rf_validacion_serie = pd.Series(
    prob_rf_ajustado_validacion,
    index=validacion.index
)

resultados_anuales_rf = []

for anio in [2023, 2024]:
    mascara_anio = (
        anio_objetivo_validacion == anio
    )

    resultado_anio = calcular_metricas(
        "Random Forest ajustado",
        str(anio),
        y_validacion.loc[mascara_anio],
        pred_rf_validacion_serie.loc[mascara_anio],
        prob_rf_validacion_serie.loc[mascara_anio]
    )

    resultado_anio["Filas"] = mascara_anio.sum()

    resultados_anuales_rf.append(
        resultado_anio
    )

tabla_anual_rf = pd.DataFrame(
    resultados_anuales_rf
)

tabla_anual_rf = tabla_anual_rf[
    [
        "Modelo",
        "Particion",
        "Filas",
        "Accuracy",
        "Balanced_accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC_AUC"
    ]
]

tabla_anual_rf[columnas_metricas] = (
    tabla_anual_rf[columnas_metricas]
    .round(4)
)

display(
    tabla_anual_rf.set_index(
        ["Modelo", "Particion"]
    )
)

Filas  Accuracy  Balanced_accuracy  \
Modelo                 Particion                                       
Random Forest ajustado 2023         260    0.4654             0.4640   
                       2024         262    0.4847             0.4859   

                                  Precision  Recall      F1  ROC_AUC  
Modelo                 Particion                                      
Random Forest ajustado 2023          0.4771  0.5530  0.5123   0.4445  
                       2024          0.4599  0.5081  0.4828   0.5054

# Importancia de las variables

La importancia muestra cuánto utilizó Random Forest cada variable para separar los casos de subida y bajada

Esto no significa que una variable sea la causa del movimiento del precio. Además, si varias variables contienen información parecida, el modelo puede repartir la importancia entre ellas porque puede utilizar cualquiera para crear divisiones similares

Este análisis me permite comprobar si el modelo depende demasiado de una sola variable o si sus decisiones están repartidas entre varias características

In [8]:
importancia_variables = pd.DataFrame({
    "Variable": variables_predictoras,
    "Importancia": modelo_rf_ajustado.feature_importances_
})

importancia_variables = (
    importancia_variables
    .sort_values(
        "Importancia",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    importancia_variables.round(4)
)

,Variable,Importancia
0,Distancia_MA5,0.0811
1,Retorno_lag_5,0.0769
2,Retorno_lag_1,0.0760
3,Distancia_MA10,0.0693
4,Volatilidad_5,0.0691
5,Volatilidad_20,0.0691
6,Retorno_lag_2,0.0679
7,Retorno_lag_3,0.0667
8,Rango_diario,0.0657
9,MACD_hist,0.0641


# Conclusión del Random Forest

El Random Forest base aprendió demasiado bien los datos de entrenamiento y llegó al 100 %, pero en validación cayó por debajo del 50 %. Esto muestra que el modelo prácticamente memorizó el entrenamiento y no consiguió funcionar bien con datos nuevos.

Después de ajustar los parámetros utilizando divisiones temporales, el sobreajuste se redujo, pero la diferencia entre entrenamiento y validación siguió siendo demasiado grande. Además, los resultados estuvieron por debajo del azar tanto en 2023 como en 2024.

Las importancias se repartieron entre varias variables y Posicion_cierre no dominó las decisiones, por lo que no se repitió el problema encontrado con Yahoo Finance. Aun así, Random Forest no mostró un rendimiento estable y, por ahora, no se considera un buen candidato para el modelo final.